In [0]:
from pyspark.sql import functions as F

df = spark.read.table("dbr_dev_ua5816bd.team_tristar_bronze.stops")

df = df.withColumn(
    'stop_id',
    F.when(
        F.trim(F.col('stop_id')) == '',
        None
    ).otherwise(
        F.trim(F.col('stop_id'))
    )
)

df = df.withColumn(
    'stop_name',
    F.when(
        F.trim(F.col('stop_name')) == '',
        None
    ).otherwise(
        F.trim(F.col('stop_name'))
    )
)

df = df.withColumn(
    'stop_lat',
    F.when(
        (F.col('stop_lat').try_cast('double') < -90) |
        (F.col('stop_lat').try_cast('double') > 90) |
        F.col('stop_lat').try_cast('double').isNull(),
        None
    ).otherwise(
        F.col('stop_lat').try_cast('double')
    )
)

df = df.withColumn(
    'stop_lat',
    F.round(F.col('stop_lat'), 6)
)

df = df.withColumn(
    'stop_lon',
    F.when(
        (F.col('stop_lon').try_cast('double') < -180) |
        (F.col('stop_lon').try_cast('double') > 180) |
        F.col('stop_lon').try_cast('double').isNull(),
        None
    ).otherwise(
        F.col('stop_lon').try_cast('double')
    )
)

df = df.withColumn(
    'stop_lon',
    F.round(F.col('stop_lon'), 6)
)

df = df.withColumn(
    'stop_code',
    F.when(
        F.trim(F.col('stop_code')) == '',
        None
    ).otherwise(
        F.trim(F.col('stop_code'))
    )
)

df = df.withColumn(
    'source',
    F.when(
        F.trim(F.col('source')) == '',
        None
    ).otherwise(
        F.trim(F.col('source'))
    )
)

df = df.withColumn(
    'source_update_date',
    F.col('source_update_date').try_cast('date')
)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "dbr_dev_ua5816bd.team_tristar_silver.stops"
)

silver_table.alias("silver").merge(
    df.alias("bronze"),
    "silver.stop_id = bronze.stop_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).whenNotMatchedBySourceDelete(
).execute()